# Demo: Catalog searches with CQL2 requests

Demo of work done as part of tickets RSPY-160 and RSPY-656.   
This shows the usage of advanced temporal filters following these specifications: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/4.+External+data+selection+policies

## 0 - Initialization

In [ ]:
import requests
import os
import pprint
import time
import pystac
from pystac import Asset, Collection, Extent, Item, SpatialExtent, TemporalExtent, ItemCollection
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
auxip_client, cadip_client, catalog_client, staging_client, *_ = init_demo()

if os.getenv("RSPY_LOCAL_MODE") == "1":
    href_cadip = "http://rs-server-cadip:8000"
    href_adgs = "http://rs-server-adgs:8000"
else:
    href_cadip = href_adgs = os.environ["RSPY_WEBSITE"]
    session.cookies.set("session", os.environ["RSPY_OAUTH2_COOKIE"])

cadip_collection_id = "cadip_sentinel1"
adgs_collection_id = "adgs"
TIMEOUT = 10
collection_description = Collection(
    id=TEST_COLLECTION,
    description=None,  # rs-client will provide a default description for us
    extent=Extent(
        spatial=SpatialExtent(bboxes=[-180.0, -90.0, 180.0, 90.0]),
        temporal=TemporalExtent([start_date, stop_date]),
    ),
)

# Init the dask cluster
from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

## 1 - Building catalog

The following code is taken from demo called "404_582_rsclient.ipynb".   
This section creates a catalog by staging data from AUXIP and CADIP stations.

In [ ]:
# Create a test collection 
collection = create_test_collection()

# Get all the items from the collection "cadip_sentinel1" found in the configuration of the CADIP station
items_collection_cadip = list(cadip_client.get_items(cadip_collection_id))
assert len(items_collection_cadip) > 0

# Request 14 items from the collection "adgs" found in the configuration of the ADGS station
items_collection_adgs = auxip_client.search(max_items = 14, collections = [adgs_collection_id])
assert len(items_collection_adgs) == 14

# Starting 2 staging processes, one from the CADIP station and one from the ADGS station
staging_resp_list = []
for items in [pystac.ItemCollection(list(items_collection_cadip)), pystac.ItemCollection(list(items_collection_adgs))]:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), TEST_COLLECTION))
    
for resp in staging_resp_list:
    staging_client.wait_for_jobs(resp, logger)

In [ ]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_items(TEST_COLLECTION))

for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")

In [ ]:
item = result[0]

# From sprin14/273_280_stac_authentication_extension.ipynb
# In this cell, we will call all the catalog GET endpoints and check that 
# the stac authentication extension is set in each JSON output value.

# We extract the GET endpoint paths from the openapi.json specification
openapi_endpoint = "/catalog/api"
openapi_response = session.get(f"{catalog_client.href_service}{openapi_endpoint}")
openapi_response.raise_for_status()
openapi = openapi_response.json()
get_endpoints = [
    key
    for key, value in openapi["paths"].items()
    if key.startswith("/catalog") and ("get" in value)
]

# Call the endpoints with and without the owner_id: prefix
if cluster_mode:
    implicit_ownerid = []
    for endpoint in get_endpoints:
         implicit_ownerid.append(endpoint)
         if "{owner_id}:{collection_id}" in endpoint:
             implicit_ownerid.append(endpoint.replace("{owner_id}:{collection_id}", "{collection_id}"))
    get_endpoints = implicit_ownerid

# Also test the POST search requests 
post_endpoints = [
    "/catalog/search",
]
search_collection = f"{catalog_client.owner_id}:{collection.id}"

# Now call each endpoint with the right method
for method, endpoint in \
    [["GET", endpoint] for endpoint in get_endpoints] + \
    [["POST", endpoint] for endpoint in post_endpoints] \
:
    # Don't call these ones, they don't implement the authentication extension
    if endpoint in [
        "/catalog/api", # openapi.json specification
        "/catalog/api.html", # swagger page
        "/catalog/docs/oauth2-redirect", # don't really know what this is
        "/catalog/conformance", # returns the conformsTo links
        # don't really know what the queryables do, but the authentication extension 
        # is not implemented for these.
        "/catalog/queryables",
        "/catalog/collections/{owner_id}:{collection_id}/queryables",
        "/catalog/collections/{collection_id}/queryables",
    ]:
        print(f"{method} {endpoint}\n{'='*(len(method+endpoint)+1)}\n(ignored)\n")
        continue

    # Replace the {owner_id}, {collection_id}, {item_id} by the right values
    endpoint = endpoint.format(
        owner_id=catalog_client.owner_id, collection_id=collection.id, item_id=item.id,
    )
    print(f"{method} {endpoint}\n{'='*(len(method+endpoint)+1)}")

    # Restrict search responses to the item staged by this notebook. Without these
    # filters, search returns items left by other demos and makes this test order-dependent.
    request_url = f"{catalog_client.href_service}{endpoint}"
    if method == "GET":
        params = (
            {"collections": search_collection, "ids": item.id}
            if endpoint == "/catalog/search"
            else None
        )
        response = session.get(request_url, params=params)
    else:
        body = (
            {"collections": [search_collection], "ids": [item.id]}
            if endpoint == "/catalog/search"
            else {}
        )
        response = session.post(request_url, json=body)
    response.raise_for_status()
    stac = response.json()
    pretty_print(stac)

    # Check extension for the root element + nested collections and items
    elements = [stac] + stac.get("collections", []) + stac.get("features", [])
    for stac in elements:

        url = "https://stac-extensions.github.io/authentication/v1.1.0/schema.json"
        properties = stac.get("properties", {})
    
        # In cluster mode, we check that the extension is implemented
        if cluster_mode:
            assert url in stac["stac_extensions"]
            assert ("auth:schemes" in stac) or  ("auth:schemes" in properties)
            for link in stac.get("links", []):
                assert link["auth:refs"] == ["apikey", "openid", "oauth2"]
            for asset in stac.get("assets", {}).values():
                assert asset["auth:refs"] == ["apikey", "openid", "oauth2"]
    
        # In local mode, we check that it is NOT implemented
        else:
            assert url not in stac.get("stac_extensions", [])
            assert ("auth:schemes" not in stac) and ("auth:schemes" not in properties)
            for link in stac.get("links", []):
                assert "auth:refs" not in link    
            for asset in stac.get("assets", {}).values():
                assert "auth:refs" not in asset

        print(f"{stac.get('id', 'Root element')} is OK") 
    print()

## 2 - Run various search requests with different filters to retrieve parts of the data

Filters are the ones described here: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/4.+External+data+selection+policies

Forked version of Pygeofilter: https://github.com/RS-PYTHON/pygeofilter

In [ ]:
# ValCover filter
# This mode gets all files that cover entirely time interval  [t0 – dt0, t1 + dt1].

valcover_filter =  {
    "op": "t_contains",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-05-27T09:44:12.509000Z", "2024-05-27T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_CONTAINS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-05-27T09:44:12.509000Z%27),TIMESTAMP(%272024-05-27T09:44:13.509000Z%27)))

params = {
    "owner_id": OWNER_ID,
    "max_items": 100,
    "collections": ["my_test_collection"],
    "stac_filter": valcover_filter
}

catalog_client.search(**params)

In [ ]:
# LatestValCover filter
# This mode gets the latest file that covers entirely time interval  [t0 – dt0, t1 + dt1]. The latest record is the one with the more recent Generation Date.

latestvalcover_filter =  {
    "op": "t_contains",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-05-27T09:44:12.509000Z", "2024-05-27T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_CONTAINS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-05-27T09:44:12.509000Z%27),TIMESTAMP(%272024-05-27T09:44:13.509000Z%27)))&sortby=-properties.created&limit=1

params = {
    "owner_id": OWNER_ID,
    "collections": ["my_test_collection"],
    "stac_filter": latestvalcover_filter,
    "sortby": [
        {
            "field": "created",
            "direction": "desc"
        }
    ],
    "max_items": 1,
}

catalog_client.search(**params)

In [ ]:
# ValIntersect filter
# This mode gets all files that cover partly time interval  [t0 – dt0, t1 + dt1].

valintersect_filter =  {
    "op": "t_intersects",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-01-21T09:44:12.509000Z", "2024-06-26T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_INTERSECTS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-01-21T09:44:12.509000Z%27),TIMESTAMP(%272024-06-26T09:44:13.509000Z%27)))

params = {
    "owner_id": OWNER_ID,
    "max_items": 100,
    "collections": ["my_test_collection"],
    "stac_filter": valintersect_filter
}

catalog_client.search(**params)

In [ ]:
# LatestValIntersect filter
# This mode gets the latest file that covers partly time interval  [t0 – dt0 , t1 + dt1]. The latest record is the one with the more recent Generation Date.

latestvalintersect_filter =  {
    "op": "t_intersects",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-01-21T09:44:12.509000Z", "2024-06-26T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_INTERSECTS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-01-21T09:44:12.509000Z%27),TIMESTAMP(%272024-06-26T09:44:13.509000Z%27)))&sortby=-properties.created&limit=1

params = {
    "owner_id": OWNER_ID,
    "collections": ["my_test_collection"],
    "stac_filter": latestvalintersect_filter,
    "sortby": [{"field": "created", "direction": "desc"}],
    "max_items": 1,
}

catalog_client.search(**params)

In [ ]:
# LatestValidity filter
# This mode gets a product with the latest Validity Start Time.

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&sortby=-properties.created&limit=1

params = {
    "owner_id": OWNER_ID,
    "collections": ["my_test_collection"],
    "sortby": [{"field": "created", "direction": "desc"}],
    "max_items": 1,
}

catalog_client.search(**params)

## 3 - Delete the catalog collection

In [ ]:
result = catalog_client.remove_collection(TEST_COLLECTION)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())